# EDA 07 — Logements Sociaux Financés à Paris
**Source** : Paris Open Data — Logements sociaux financés  
**Bronze path** : `data/bronze/social_housing/part-0.parquet`  
**Fichier brut** : `data/bronze/logement_sociaux/logements-sociaux-finances-a-paris.parquet`

## Schéma Bronze (standardisé)
| Colonne | Type | Description |
|---|---|---|
| `arrondissement` | int | 1-20 |
| `annee` | int | Année de financement |
| `nombre_logements` | int | Nombre de logements financés |

## Schéma Brut (logements-sociaux-finances-a-paris.parquet)
| Colonne | Type | Description |
|---|---|---|
| `id_livraison` | str | Identifiant du programme |
| `adresse_programme` | str | Adresse |
| `nb_logmt_total` | int | Total logements |
| `nb_plai` / `nb_plus` / `nb_pls` | int | Logements par type de financement |
| `mode_real` | str | Mode de réalisation |
| `nature_programme` | str | Type de programme |


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")

path_std  = f"{BRONZE_ROOT}/social_housing/part-0.parquet"
path_raw  = f"{BRONZE_ROOT}/logement_sociaux/logements-sociaux-finances-a-paris.parquet"

df_std = pd.read_parquet(path_std) if os.path.exists(path_std) else pd.DataFrame()
df_raw = pd.read_parquet(path_raw) if os.path.exists(path_raw) else pd.DataFrame()

if df_std.empty and df_raw.empty:
    print("⚠️  Aucun fichier Bronze disponible")
else:
    if not df_std.empty:
        print(f"Bronze standardisé : {df_std.shape}")
        display(df_std.head(3))
    if not df_raw.empty:
        print(f"\nBronze brut : {df_raw.shape}")
        display(df_raw.head(3))


In [ ]:
# ── Analyse du dataset brut ──────────────────────────────────────────────────
if not df_raw.empty:
    df_raw["annee"] = pd.to_datetime(df_raw["annee"], errors="coerce").dt.year
    
    print("Modes de réalisation :")
    display(df_raw["mode_real"].value_counts().head(10))
    print("\nNature de programme :")
    display(df_raw["nature_programme"].value_counts())
    print(f"\nTotal logements : {df_raw['nb_logmt_total'].sum():,}")
    print(f"Dont PLAI : {df_raw['nb_plai'].sum():,} ({df_raw['nb_plai'].sum()/df_raw['nb_logmt_total'].sum():.1%})")
    print(f"Dont PLUS : {df_raw['nb_plus'].sum():,} ({df_raw['nb_plus'].sum()/df_raw['nb_logmt_total'].sum():.1%})")
    print(f"Dont PLS  : {df_raw['nb_pls'].sum():,} ({df_raw['nb_pls'].sum()/df_raw['nb_logmt_total'].sum():.1%})")


In [ ]:
# ── Évolution annuelle ───────────────────────────────────────────────────────
if not df_std.empty:
    df_time = df_std.groupby("annee")["nombre_logements"].sum().reset_index()
    df_time = df_time.dropna(subset=["annee"])
    df_time["annee"] = df_time["annee"].astype(int)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(df_time["annee"], df_time["nombre_logements"],
           color=plt.cm.Blues(df_time["nombre_logements"] / df_time["nombre_logements"].max()))
    ax.set_xlabel("Année")
    ax.set_ylabel("Logements financés")
    ax.set_title("Évolution annuelle des logements sociaux financés à Paris")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    plt.tight_layout()
    plt.show()
    print(f"Années disponibles : {sorted(df_time['annee'].tolist())}")
elif not df_raw.empty:
    df_time = df_raw.groupby("annee")["nb_logmt_total"].sum().dropna().astype(int)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(df_time.index.astype(int), df_time.values,
           color=plt.cm.Blues(df_time.values / df_time.max()))
    ax.set_title("Évolution annuelle (source brute)")
    ax.set_xlabel("Année")
    ax.set_ylabel("Logements financés")
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Logements par arrondissement ─────────────────────────────────────────────
df_ref = df_std if not df_std.empty else (df_raw.rename(columns={"arrdt":"arrondissement","nb_logmt_total":"nombre_logements"}) if not df_raw.empty else pd.DataFrame())

if not df_ref.empty:
    arr_total = (
        df_ref.groupby("arrondissement")["nombre_logements"]
        .sum()
        .sort_values(ascending=False)
    )
    
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = plt.cm.YlGnBu(arr_total.values / arr_total.max())
    bars = ax.bar(arr_total.index.astype(str), arr_total.values, color=colors)
    ax.set_xlabel("Arrondissement")
    ax.set_ylabel("Total logements sociaux financés")
    ax.set_title("Logements sociaux financés par arrondissement (cumulé)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
    plt.tight_layout()
    plt.show()
    print("Top 5 arrondissements :")
    print(arr_total.head(5).to_string())


In [ ]:
# ── Types de financement (données brutes) ────────────────────────────────────
if not df_raw.empty:
    df_raw_clean = df_raw.copy()
    df_raw_clean["annee"] = pd.to_datetime(df_raw_clean["annee"], errors="coerce").dt.year
    
    by_arr = df_raw_clean.groupby("arrdt")[["nb_plai","nb_plus","nb_pluscd","nb_pls"]].sum()
    
    fig, ax = plt.subplots(figsize=(14, 7))
    bottom = np.zeros(len(by_arr))
    colors_fin = {"nb_plai":"#F44336","nb_plus":"#FF9800","nb_pluscd":"#FFC107","nb_pls":"#4CAF50"}
    labels = {"nb_plai":"PLAI (très social)","nb_plus":"PLUS (social)","nb_pluscd":"PLUS-CD","nb_pls":"PLS (intermédiaire)"}
    
    for col in ["nb_plai","nb_plus","nb_pluscd","nb_pls"]:
        ax.bar(by_arr.index.astype(str), by_arr[col], bottom=bottom,
               label=labels[col], color=colors_fin[col])
        bottom += by_arr[col].values
    
    ax.set_xlabel("Arrondissement")
    ax.set_ylabel("Logements")
    ax.set_title("Répartition par type de financement par arrondissement")
    ax.legend(loc="upper right")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
    plt.tight_layout()
    plt.show()


## Conclusions EDA
- Les 13e, 19e et 20e arrondissements concentrent le plus grand nombre de logements sociaux financés historiquement.
- Les logements PLUS (social) dominent le mix, suivis des PLS (intermédiaires).
- L'activité de financement montre des pics sur certaines années de politique municipale active.
- Cet indicateur entre dans le Gold `arrondissement_summary` comme `nombre_logements_sociaux`.
